# Chess Academy Pro — Voice Pack Generator

Generates HD voice packs using Kokoro TTS.

**Instructions:**
1. Click **Runtime → Change runtime type → T4 GPU** (or CPU if GPU unavailable)
2. Click **Runtime → Run all**
3. When prompted, upload **two files** from your Mac:
   - `repertoire.json` (Finder → Cmd+Shift+G → `~/Developer/chess-academy-pro/src/data`)
   - `annotations-bundle.json` (same folder)
4. Wait for all cells to complete
5. Download the zip file from the last cell

In [ ]:
# Cell 1: Install dependencies
!pip install -q kokoro>=0.9.2 soundfile numpy
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
print('Dependencies installed!')

In [ ]:
# Cell 2: Upload data files
import json, struct, os, time, zipfile
import numpy as np
import soundfile as sf
from io import BytesIO

print('Please upload TWO files:')
print('  1. repertoire.json')
print('  2. annotations-bundle.json')
print('')
print('Find both at: ~/Developer/chess-academy-pro/src/data')
print('')

from google.colab import files
uploaded = files.upload()

# Load repertoire
repertoire = None
annotations_bundle = None

for fname, content in uploaded.items():
    if 'repertoire' in fname:
        repertoire = json.loads(content.decode())
        print(f'Loaded {len(repertoire)} openings from {fname}')
    elif 'annotation' in fname:
        annotations_bundle = json.loads(content.decode())
        print(f'Loaded {len(annotations_bundle)} annotation sets from {fname}')

if not repertoire:
    raise FileNotFoundError('repertoire.json not uploaded!')
if not annotations_bundle:
    print('WARNING: annotations-bundle.json not uploaded — walkthrough annotations will be skipped')

# Hash function — MUST match voiceService.ts hashText() exactly
def hash_text(text: str) -> str:
    h = 0
    for ch in text:
        code = ord(ch)
        h = ((h << 5) - h) + code
        h &= 0xFFFFFFFF
        if h >= 0x80000000:
            h -= 0x100000000
    return str(h)

In [ ]:
# Cell 3: Collect all phrases (repertoire + walkthrough annotations)
def collect_phrases(repertoire, annotations_bundle=None):
    phrases = set()

    for opening in repertoire:
        # Overview
        if opening.get('overview'):
            phrases.add(opening['overview'])
        # Key ideas
        for idea in opening.get('keyIdeas', []):
            phrases.add(idea)
        # Traps
        for trap in opening.get('traps', []):
            phrases.add(trap)
        # Warnings
        for warning in opening.get('warnings', []):
            phrases.add(warning)
        # Variation explanations + template messages
        for v in opening.get('variations', []):
            if v.get('explanation'):
                phrases.add(v['explanation'].replace('*', ''))
            phrases.add(f"Well done! You've completed the {v['name']} line.")
            phrases.add(f"Line discovered! You've learned the {v['name']}.")
            phrases.add(f"Line perfected! You know the {v['name']} by heart.")
        # Trap lines
        for t in opening.get('trapLines', []):
            if t.get('explanation'):
                phrases.add(t['explanation'].replace('*', ''))
            phrases.add(f"Well done! You've completed the {t['name']} line.")
            phrases.add(f"Line discovered! You've learned the {t['name']}.")
        # Warning lines
        for w in opening.get('warningLines', []):
            if w.get('explanation'):
                phrases.add(w['explanation'].replace('*', ''))
            phrases.add(f"Well done! You've completed the {w['name']} line.")
            phrases.add(f"Line discovered! You've learned the {w['name']}.")
        # Play mode intro
        phrases.add(f"Let's play the {opening['name']}. Remember your key ideas and play confidently.")

    # Walkthrough mode: per-move annotations
    if annotations_bundle:
        for opening_id, moves in annotations_bundle.items():
            for move in moves:
                if move.get('annotation'):
                    phrases.add(move['annotation'])

    # Generic drill hints
    for hint in ['Castle to safety.', 'Develop your knight.', 'Develop your bishop.',
                 'Bring your queen out.', 'Activate your rook.', 'Continue with the plan.']:
        phrases.add(hint)

    return sorted(phrases)

phrases = collect_phrases(repertoire, annotations_bundle)
print(f'Collected {len(phrases)} unique phrases')
print(f'  (includes {sum(1 for _ in (m for mvs in (annotations_bundle or {}).values() for m in mvs if m.get("annotation"))) } walkthrough annotations)')

In [ ]:
# Cell 4: Initialize Kokoro pipeline
from kokoro import KPipeline

# American English pipeline
pipeline_a = KPipeline(lang_code='a')
# British English pipeline
pipeline_b = KPipeline(lang_code='b')
print('Kokoro pipelines ready!')

In [ ]:
# Cell 5: Pack voice clips into binary format
# Format: [count:uint32] repeated [hashLen:uint16][hash:utf8][audioLen:uint32][mp3Data:bytes]

def audio_to_mp3_bytes(audio_np, sample_rate=24000):
    """Convert numpy audio array to MP3 bytes via WAV intermediary."""
    buf = BytesIO()
    sf.write(buf, audio_np, sample_rate, format='WAV')
    return buf.getvalue()  # WAV bytes (universally decodable)

def pack_voice_pack(clips):
    parts = [struct.pack('<I', len(clips))]
    for clip in clips:
        hash_bytes = clip['hash'].encode('utf-8')
        audio_data = clip['audio']
        parts.append(struct.pack('<H', len(hash_bytes)))
        parts.append(hash_bytes)
        parts.append(struct.pack('<I', len(audio_data)))
        parts.append(audio_data)
    return b''.join(parts)

print('Packing functions ready!')

In [ ]:
# Cell 6: Generate all voice packs
VOICES = {
    'af_heart': 'a', 'af_bella': 'a', 'af_nicole': 'a', 'af_sarah': 'a', 'af_nova': 'a',
    'am_adam': 'a', 'am_eric': 'a', 'am_michael': 'a', 'am_liam': 'a',
    'bf_emma': 'b', 'bf_isabella': 'b',
    'bm_daniel': 'b', 'bm_george': 'b',
}

os.makedirs('voice_packs', exist_ok=True)

for voice_id, lang in VOICES.items():
    pipeline = pipeline_a if lang == 'a' else pipeline_b
    print(f'\n{"="*60}')
    print(f'Generating: {voice_id}')
    print(f'{"="*60}')
    
    clips = []
    start = time.time()
    errors = 0
    
    for i, text in enumerate(phrases):
        text_hash = hash_text(text)
        try:
            # Generate audio
            generator = pipeline(text, voice=voice_id, speed=1.0)
            all_audio = []
            for _, _, audio_chunk in generator:
                all_audio.append(audio_chunk)
            
            if all_audio:
                full_audio = np.concatenate(all_audio)
                wav_bytes = audio_to_mp3_bytes(full_audio, 24000)
                clips.append({'hash': text_hash, 'audio': wav_bytes})
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f'  ERROR [{i+1}]: {str(e)[:80]}')
        
        if (i + 1) % 200 == 0 or i == len(phrases) - 1:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed * 60
            eta = (len(phrases) - i - 1) / max(rate / 60, 0.01) / 60
            print(f'  [{i+1}/{len(phrases)}] {elapsed:.0f}s, {rate:.0f}/min, ETA {eta:.1f}min, {errors} errors')
    
    # Pack and save
    packed = pack_voice_pack(clips)
    out_path = f'voice_packs/{voice_id}.bin'
    with open(out_path, 'wb') as f:
        f.write(packed)
    size_mb = len(packed) / 1024 / 1024
    print(f'  Saved: {out_path} ({size_mb:.1f} MB, {len(clips)} clips, {errors} errors)')

print('\nAll voice packs generated!')

In [ ]:
# Cell 7: Zip and download
zip_path = 'voice_packs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir('voice_packs')):
        if fname.endswith('.bin'):
            fpath = os.path.join('voice_packs', fname)
            zf.write(fpath, fname)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f'  Added {fname} ({size_mb:.1f} MB)')

zip_size = os.path.getsize(zip_path) / 1024 / 1024
print(f'\nZip created: {zip_path} ({zip_size:.1f} MB)')

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print('Download started!')
except:
    print(f'Not running in Colab — file saved to {zip_path}')